In [5]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)


Setup complete. pandas 2.2.3


In [6]:
import pandas as pd

df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000]
})

print(df)


     city    size  age  income
0    pune   small   25   38000
1   delhi   large   41   92000
2  mumbai  medium   33   55000
3    pune  medium   29   47000
4   delhi   small   52  120000
5  mumbai   large   38   76000
6    pune   large   46   88000
7   delhi   small   22   41000


1. Encoding categorical data

1A. ONE-HOT ENCODING (for UNORDERED categories)

In [7]:
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe

,city_delhi,city_mumbai,city_pune
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0
6,0,0,1
7,1,0,0


1B. LABEL / ORDINAL ENCODING (for ORDERED categories)

In [8]:
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]



,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


LAB EXERCISE 1 — Encode the columns


1. One-hot encode city

In [9]:
df = pd.get_dummies(df, columns=['city'])


2. Ordinal encode size


In [10]:
size_order = {'small': 1, 'medium': 2, 'large': 3}
df['size'] = df['size'].map(size_order)


3. Explanation

One-hot encoding is wrong for size because size has a natural order.
 Ordinal encoding is wrong for city because cities have no natural order.


2. Feature scaling

2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES

In [11]:
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])


        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0


2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)

In [12]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)
m = MinMaxScaler().fit_transform(num)
print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))



Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


LAB EXERCISE 2 — Scale a feature


1. Standardise


In [14]:
from sklearn.preprocessing import StandardScaler

income = df[['income']]

standard_scaler = StandardScaler()
income_standard = standard_scaler.fit_transform(income)

print(income_standard.mean())
print(income_standard.std())


-2.7755575615628914e-17
1.0


2. Min-Max scale

In [15]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
income_minmax = minmax_scaler.fit_transform(income)

print(income_minmax.min())
print(income_minmax.max())


0.0
1.0000000000000002


3. Scaling explanation

Needs scaling: income
Doesn't need it: one-hot encoded columns


3. Avoiding data leakage


3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY

In [18]:
df['bought'] = [0, 1, 1, 0, 1, 1, 1, 0]
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))


train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


LAB EXERCISE 3 — Spot the leak


In [19]:
# 1a. Why is fitting the scaler on ALL data before splitting a form of data leakage?
wrong_mean = StandardScaler().fit(X).mean_


In [20]:
# 1b. Why should the scaler be fitted on the TRAIN data only?
right_mean = StandardScaler().fit(X_train).mean_


2. Why is fitting the scaler on ALL data a problem?





 Because the scaler learns information from the test data, causing data leakage.


4. Feature engineering

4A. COMBINE & BIN EXISTING COLUMNS

In [21]:

fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]



,age,income,income_per_age,age_group
0,25,38000,1520.0,young
1,41,92000,2244.0,mid
2,33,55000,1667.0,mid
3,29,47000,1621.0,young
4,52,120000,2308.0,senior
5,38,76000,2000.0,mid
6,46,88000,1913.0,senior
7,22,41000,1864.0,young


4B. EXTRACT FROM A DATE

In [23]:
dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5


LAB EXERCISE 4 — Engineer two features


In [25]:
ex = df.copy()

# 1. How can we create a high_earner flag where income is greater than the median?
ex['high_earner'] = ex['income'] > ex['income'].median()


In [26]:
# 2. How can we divide income into 3 buckets using pd.cut?
ex['income_bucket'] = pd.cut(ex['income'], bins=3, labels=['low', 'medium', 'high'])


In [27]:
# 3. How can we display income along with the new columns?
print(ex[['income', 'high_earner', 'income_bucket']])


   income  high_earner income_bucket
0   38000        False           low
1   92000         True        medium
2   55000        False           low
3   47000        False           low
4  120000         True          high
5   76000         True        medium
6   88000         True        medium
7   41000        False           low


5. The preprocessing pipeline

5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']


pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)



Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])


 5B. FIT THE WHOLE THING IN ONE CALL

In [34]:
import pandas as pd

df = pd.DataFrame({
    'city': ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size': ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age': [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 1, 0, 1, 1, 1, 0]
})
num_cols = ['age', 'income']
cat_cols = ['city', 'size']

X = df[num_cols + cat_cols]
y = df['bought']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)


LAB EXERCISE 5 — Build your own pipeline


In [46]:
# 1. How can we create a pipeline using MinMaxScaler and LogisticRegression?
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', LogisticRegression())
])


In [47]:
# 2. How can we split the data and fit the pipeline on the training data?
from sklearn.model_selection import train_test_split

Xn = df[['age', 'income']]
yn = df['bought']

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    Xn, yn, test_size=0.25, random_state=0
)

pipe.fit(Xn_train, yn_train)


Pipeline(steps=[('scaler', MinMaxScaler()), ('model', LogisticRegression())])

In [48]:
# 3. How can we calculate and print the test accuracy?
Xn_test = Xn_test
yn_test = yn_test

acc = pipe.score(Xn_test, yn_test)
print('Test accuracy:', round(acc, 2))


Test accuracy: 0.5
